# VEILINK 六轴机械臂
# 一、舵机编号与实体标记

用途：在 STS3215 舵机装入机械臂之前，逐台检查通信、向舵机写入唯一 ID，并立即粘贴关节标签。

本项目的默认编号是：六个运动关节使用 **J1～J6 / ID1～ID6**；如果夹爪也由 STS3215 驱动，则使用 **Gripper / ID7**。机械臂电机由下到上命名。

> 本 Notebook 只设置舵机 ID 和通信波特率，不设置机械零位，不测量关节极限，也不会主动命令舵机转动。零位、方向和限位必须在机械装配完成后单独校准。

## 0. 操作前必须阅读

1. 在 VS Code 或 Jupyter 右上角选择已经配置好的 `lerobot` Python kernel。
2. 核对舵机铭牌和供电电压。STS3215 有不同电压版本，不要只根据系列名称接电。
3. 舵机必须使用符合其规格的外部电源；USB 主要负责通信，不应依靠 USB 给多个舵机供电。
4. **写 ID 时，控制板上一次只能连接一台舵机。** 多台全新舵机可能使用相同出厂 ID，同时连接会造成通信冲突或被写成相同 ID。
5. 接线、拔线或更换舵机前先关闭舵机电源。不要带电插拔 TTL 总线插头。
6. 确认 GND、VCC、TTL 信号线方向正确，并确保控制板与舵机电源共地。
7. 如果出现发热、异味、异常响声或线缆发烫，立即断电。

推荐准备七张标签：`J1-ID1`、`J2-ID2`、`J3-ID3`、`J4-ID4`、`J5-ID5`、`J6-ID6`、`GRIPPER-ID7`。若本次不使用电动夹爪，只准备前六张。

## 1. 检查 LeRobot 环境

运行下一格。请确认当前 Kernel 为 `lerobot` 环境，且核心模块均显示 `OK`。本 Notebook 按本项目当前安装的 LeRobot 0.5.x API 编写。

In [ ]:
import importlib.util
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

print("Python executable:", Path(sys.executable).name)
print("Python version:", sys.version.split()[0])

required_modules = ["lerobot", "serial", "serial.tools.list_ports", "ipykernel"]
missing_modules = []
for module_name in required_modules:
    ok = importlib.util.find_spec(module_name) is not None
    print(f"{module_name:26s}", "OK" if ok else "MISSING")
    if not ok:
        missing_modules.append(module_name)

if missing_modules:
    raise RuntimeError(f"当前 kernel 缺少模块：{missing_modules}。请切换到 lerobot 环境。")

try:
    lerobot_version = version("lerobot")
except PackageNotFoundError:
    lerobot_version = "unknown"

print("LeRobot version:", lerobot_version)
if lerobot_version != "unknown" and not lerobot_version.startswith("0.5."):
    print("提示：当前版本不是已验证的 0.5.x；若后续出现 API 错误，请先核对 LeRobot 版本。")

## 2. 查找控制板串口

只把舵机控制板通过 USB 接到电脑，运行下一格查看串口。Windows 通常显示为 `COM3`、`COM4` 等。

如果没有检测到串口：

- 检查 USB 线是否支持数据传输；
- 在设备管理器的“端口（COM 和 LPT）”中检查驱动；
- 关闭可能占用串口的其他串口助手或 Notebook；
- 重新插拔控制板，再运行本格。

In [ ]:
from serial.tools import list_ports

ports = list(list_ports.comports())
if not ports:
    print("没有检测到串口。请检查 USB 数据线、驱动和控制板连接。")
else:
    print("检测到以下串口：")
    for index, port in enumerate(ports, start=1):
        print(f"{index}. {port.device:8s} | {port.description} | {port.hwid}")

## 3. 填写端口并确认编号方案

把 `PORT` 改成上一步检测到的实际串口。

`INCLUDE_GRIPPER` 的含义：

- `True`：六轴机械臂之外，还有一台 STS3215 驱动夹爪，共七台舵机；
- `False`：本次只设置 J1～J6 六台关节舵机。

这里的编号按照运动学模型定义：J1 为基座回转，J6 为末端回转。

In [ ]:
# TODO：改成你的控制板实际串口，例如 COM3。
PORT = "COM3"

# 如果夹爪也使用一台 STS3215，保持 True；否则改为 False。
INCLUDE_GRIPPER = True

MOTOR_PLAN = {
    "J1": {"id": 1, "label": "J1-ID1", "function": "基座回转"},
    "J2": {"id": 2, "label": "J2-ID2", "function": "肩部俯仰"},
    "J3": {"id": 3, "label": "J3-ID3", "function": "肘部俯仰"},
    "J4": {"id": 4, "label": "J4-ID4", "function": "前臂回转"},
    "J5": {"id": 5, "label": "J5-ID5", "function": "腕部俯仰"},
    "J6": {"id": 6, "label": "J6-ID6", "function": "末端回转"},
}

if INCLUDE_GRIPPER:
    MOTOR_PLAN["gripper"] = {"id": 7, "label": "GRIPPER-ID7", "function": "夹爪开合"}

ids = [item["id"] for item in MOTOR_PLAN.values()]
if len(ids) != len(set(ids)):
    raise ValueError("编号方案中存在重复 ID。")
if any(not 0 <= motor_id <= 253 for motor_id in ids):
    raise ValueError("STS3215 ID 必须位于 0～253。")

print("PORT =", PORT)
print("\n本次编号方案：")
for joint, item in MOTOR_PLAN.items():
    print(f"{joint:8s} -> ID {item['id']} -> 标签 {item['label']:12s} -> {item['function']}")

## 4. 加载舵机总线并定义安全辅助函数

`setup_motor()` 将在当前串口搜索单台舵机，关闭其扭矩，然后写入目标 ID 和 LeRobot 默认波特率。它不会发送目标位置，不会主动转动舵机。

In [ ]:
from lerobot.motors import Motor, MotorNormMode
from lerobot.motors.feetech import FeetechMotorsBus

MODEL_NAME = "sts3215"

def make_single_motor_bus(joint: str) -> FeetechMotorsBus:
    """创建只包含目标舵机的总线对象，避免校验尚未连接的其他舵机。"""
    if joint not in MOTOR_PLAN:
        raise KeyError(f"未知关节 {joint!r}，可选值：{list(MOTOR_PLAN)}")
    target_id = MOTOR_PLAN[joint]["id"]
    norm_mode = MotorNormMode.RANGE_0_100 if joint == "gripper" else MotorNormMode.DEGREES
    return FeetechMotorsBus(
        port=PORT,
        motors={joint: Motor(target_id, MODEL_NAME, norm_mode)},
    )

def mark_one_motor(joint: str, *, initial_baudrate=None, initial_id=None) -> dict:
    """
    给当前唯一连接的舵机写入指定关节 ID，并回读验证。

    initial_baudrate/initial_id 默认留空，让 LeRobot 自动搜索。
    只有自动搜索失败且你确定舵机原始参数时才填写它们。
    """
    item = MOTOR_PLAN[joint]
    bus = make_single_motor_bus(joint)
    try:
        print(f"开始设置 {joint}：目标 ID={item['id']}，标签={item['label']}")
        bus.setup_motor(joint, initial_baudrate=initial_baudrate, initial_id=initial_id)

        model_number = bus.ping(joint, num_retry=2, raise_on_error=True)
        present_position = bus.read("Present_Position", joint, normalize=False, num_retry=2)
        present_voltage = bus.read("Present_Voltage", joint, normalize=False, num_retry=2)
        present_temperature = bus.read("Present_Temperature", joint, normalize=False, num_retry=2)

        result = {
            "joint": joint,
            "id": item["id"],
            "label": item["label"],
            "model_number": model_number,
            "present_position_raw": present_position,
            "present_voltage_raw": present_voltage,
            "present_temperature_raw": present_temperature,
        }
        print("写入并回读成功：", result)
        print(f"现在请断开舵机电源，并立即粘贴标签：{item['label']}")
        return result
    finally:
        if bus.is_connected:
            bus.disconnect(disable_torque=True)

print("辅助函数加载完成。当前步骤不会访问串口，也不会修改舵机。")

## 5. 逐台设置并标记

每台舵机严格重复以下流程：

1. 关闭舵机电源。
2. 拔下上一台舵机，控制板上只连接**当前这一台**。
3. 检查插头方向和供电电压，然后通电。
4. 把 `TARGET_JOINT` 改成当前标签，例如第一次使用 `J1`。
5. 运行代码格，等待“写入并回读成功”。
6. 关闭电源，粘贴程序提示的标签。

In [ ]:
# TODO：每完成一台后修改为下一个关节：J1、J2、J3、J4、J5、J6、gripper（可选）。
TARGET_JOINT = "J1"


# 自动搜索通常无需填写。只有已知舵机原 ID/波特率且自动搜索失败时才修改。
INITIAL_BAUDRATE = None
INITIAL_ID = None


last_result = mark_one_motor(
    TARGET_JOINT,
    initial_baudrate=INITIAL_BAUDRATE,
    initial_id=INITIAL_ID,
)
last_result

## 6. 全部完成后：整条总线验证

只有所有舵机均已单独写入不同 ID 并粘贴标签后，才执行本节。

操作步骤：

1. 关闭电源，把已编号舵机按 TTL 总线方式串联起来。
2. 再次检查供电能力、极性和共地。
3. 通电，但不要安装连杆或给舵盘加载。
4. 将 `RUN_FULL_BUS_VERIFY` 改成 `True` 并运行。

本节只进行握手和状态回读，不发送目标位置。成功时，每个规划 ID 应恰好返回一行数据。

In [ ]:
RUN_FULL_BUS_VERIFY = False

def make_full_bus() -> FeetechMotorsBus:
    motors = {}
    for joint, item in MOTOR_PLAN.items():
        norm_mode = MotorNormMode.RANGE_0_100 if joint == "gripper" else MotorNormMode.DEGREES
        motors[joint] = Motor(item["id"], MODEL_NAME, norm_mode)
    return FeetechMotorsBus(port=PORT, motors=motors)

if not RUN_FULL_BUS_VERIFY:
    print("尚未执行整条总线验证。全部舵机编号完成后，将 RUN_FULL_BUS_VERIFY 改为 True。")
else:
    full_bus = make_full_bus()
    try:
        full_bus.connect(handshake=True)
        positions = full_bus.sync_read("Present_Position", normalize=False, num_retry=2)
        voltages = full_bus.sync_read("Present_Voltage", normalize=False, num_retry=2)
        temperatures = full_bus.sync_read("Present_Temperature", normalize=False, num_retry=2)

        print("全部规划舵机通信成功：")
        for joint, item in MOTOR_PLAN.items():
            print(
                f"{joint:8s} ID={item['id']:3d} "
                f"position_raw={positions[joint]!s:>6s} "
                f"voltage_raw={voltages[joint]!s:>4s} "
                f"temperature_raw={temperatures[joint]!s:>4s} "
                f"label={item['label']}"
            )
    finally:
        if full_bus.is_connected:
            full_bus.disconnect(disable_torque=True)

## 7. 完成检查表

完成本阶段后，应满足：

- J1 已设置为 ID1，并粘贴 `J1-ID1`；
- J2 已设置为 ID2，并粘贴 `J2-ID2`；
- J3 已设置为 ID3，并粘贴 `J3-ID3`；
- J4 已设置为 ID4，并粘贴 `J4-ID4`；
- J5 已设置为 ID5，并粘贴 `J5-ID5`；
- J6 已设置为 ID6，并粘贴 `J6-ID6`；
- 如果使用电动夹爪，夹爪已设置为 ID7，并粘贴 `GRIPPER-ID7`；
- 所有舵机串联后，整条总线验证全部通过；
- 没有重复 ID、掉线、异常温升或供电异常。

完成后先断电，再按标签顺序装配机械臂。下一阶段应在装配前把舵机移动到统一参考位置，并在装配后建立电机位置、机械零位和 DH 零位之间的映射。

## 8. 常见问题

**找不到任何舵机**  
确认外部电源已打开、TTL 插头方向正确、控制板与电源共地、串口没有被其他程序占用。尝试重新插拔 USB 并重启 kernel。

**自动搜索很慢**  
LeRobot 会尝试多个常见波特率和 ID。若你明确知道舵机当前波特率和 ID，可在逐台设置格中填写 `INITIAL_BAUDRATE` 和 `INITIAL_ID`；不确定时不要猜。

**写入成功但重新连接失败**  
先断电重启舵机，只连接这一台，再用相同 `TARGET_JOINT` 重试回读。检查是否错误选择了关节目标或串口。

**串联后某个 ID 掉线**  
先断电，把对应标签的舵机单独连接并验证。如果单独正常，重点检查串联线缆、接头压接、电源压降和总线拓扑。

**舵机在本阶段发生转动**  
立即断电。本 Notebook 不发送目标位置；检查是否有其他控制程序占用串口或运行了其他 Notebook。重启 kernel，关闭其他控制程序后再排查。

**是否现在设置中位和机械限位？**  
不要。本阶段只编号。中位安装、关节正方向、编码器零偏和软限位必须结合实际装配结构完成。